# Vector Store Notebook

This notebook builds the FAISS vector store from the knowledge base and tests retrieval with sample prompts using the **Retriever** class with hybrid scoring.

## Purpose
1.  **Load Knowledge Base**: Use `KnowledgeBase` class to load entries from `data/knowledge_base/`.
2.  **Build Vector Index**: Index entries using `KnowledgeBase.index_entries()`.
3.  **Use Retriever**: Import and use the `Retriever` class from `src.rag` with hybrid scoring.
4.  **Test Retrieval**: Run sample prompts and display top-5 retrieved entries.
5.  **Save Index**: Persist the vector store for later use.

## Hybrid Scoring (from src.rag.retriever)
**Hybrid Score = Semantic Score + Topic Boost**
- Semantic: Embedding similarity (0-1)
- Topic Boost: +0.15 per matching topic keyword in query


In [1]:
# Setup and Imports
import sys
import json
from pathlib import Path
import numpy as np

# Add project root to path
project_root = Path("..").resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from config import get_config, get_config_loader

# Import RAG components from src
from src.rag import (
    VectorStore, 
    EmbeddingModel, 
    KnowledgeBase,
    Retriever,
    extract_query_keywords,
    DEFAULT_TOPIC_BOOST,
)

print("✅ Imports successful!")
print(f"Default Topic Boost: {DEFAULT_TOPIC_BOOST}")

✅ Imports successful!
Default Topic Boost: 0.15


## 1. Initialize Knowledge Base and Load Entries

Using the `KnowledgeBase` class to properly load entries so they are available for search.

In [2]:
# Define paths
KNOWLEDGE_BASE_DIR = project_root / "data" / "knowledge_base"
VECTOR_INDEX_PATH = project_root / "data" / "models" / "vector_index"

# Initialize Knowledge Base with the path
knowledge_base = KnowledgeBase(
    knowledge_base_path=KNOWLEDGE_BASE_DIR,
)

# Load entries from directory (this populates entry_by_id)
num_loaded = knowledge_base.load_from_directory()

print(f"\n✅ Loaded {num_loaded} entries from knowledge base.")
print(f"Entries indexed by ID: {len(knowledge_base.entry_by_id)}")

2026-03-26 00:50:56.437 | INFO     | config_loader:load:101 - ✅ Loaded configuration from D:\University\Uni\Huawei ICT Compeition\Cirq-RAG-Code-Assistant\config\config.json
2026-03-26 00:50:56.439 | DEBUG    | config_loader:create_directories:243 - Created all necessary directories
2026-03-26 00:50:56.599 | INFO     | src.rag.embeddings:_init_aws:124 - Using AWS Bedrock Embeddings: amazon.nova-2-multimodal-embeddings-v1:0, dim=1024
2026-03-26 00:50:56.600 | INFO     | src.rag.vector_store:_init_faiss:139 - Initialized FAISS index
2026-03-26 00:50:56.601 | INFO     | src.rag.vector_store:__init__:120 - Initialized VectorStore with faiss backend
2026-03-26 00:50:56.601 | INFO     | src.rag.vector_store:__init__:121 - Embedding dimension: 1024
2026-03-26 00:50:56.602 | INFO     | src.rag.knowledge_base:__init__:100 - Initialized KnowledgeBase with 0 entries
2026-03-26 00:50:56.603 | INFO     | src.rag.knowledge_base:load_from_jsonl:116 - Loading knowledge base from D:\University\Uni\Huawe


✅ Loaded 140 entries from knowledge base.
Entries indexed by ID: 140


## 2. View Sample Entries

In [3]:
# Display sample entries
print("Sample entries:")
for i, entry in enumerate(knowledge_base.entries[:3]):
    print(f"\n--- Entry {i+1} ---")
    print(f"ID: {entry.get('id', 'N/A')}")
    print(f"Difficulty: {entry.get('difficulty', 'N/A')}")
    print(f"Topics: {entry.get('topics', [])}")
    print(f"Task: {entry.get('task', 'N/A')[:100]}...")

Sample entries:

--- Entry 1 ---
ID: design_basic_adder_template
Difficulty: advanced
Topics: ['arithmetic', 'adder', 'toffoli']
Task: Design a reusable Cirq gate or function implementing a ripple-carry adder for n-bit integers using T...

--- Entry 2 ---
ID: design_basic_adder_template_v2
Difficulty: advanced
Topics: ['arithmetic', 'adder']
Task: Show how to use the RippleCarryAdder gate to add two specific 3-bit classical numbers by initializin...

--- Entry 3 ---
ID: design_bb84_round
Difficulty: intermediate
Topics: ['bb84', 'cryptography', 'basis_encoding']
Task: Write a Cirq circuit that implements one round of the BB84 protocol on a single qubit: Alice chooses...


## 3. Index Entries in Vector Store

Using `KnowledgeBase.index_entries()` to generate embeddings and build the vector store.

In [4]:
# Index all entries (generates embeddings and adds to vector store)
print("Indexing entries... (this may take a moment)")
knowledge_base.index_entries(batch_size=16)

print(f"\n✅ Vector store size: {knowledge_base.vector_store.size()}")

Indexing entries... (this may take a moment)


2026-03-26 00:50:56.627 | INFO     | src.rag.knowledge_base:index_entries:181 - Indexing 140 entries in vector store
2026-03-26 00:50:56.628 | DEBUG    | src.rag.embeddings:encode:227 - Generating embeddings for 140 texts (provider=aws)
2026-03-26 01:03:36.674 | DEBUG    | src.rag.vector_store:add:204 - Added 140 embeddings to vector store
2026-03-26 01:03:36.675 | INFO     | src.rag.knowledge_base:index_entries:248 - ✅ Indexed 140 entries



✅ Vector store size: 140


## 4. Initialize Retriever with Hybrid Scoring

Using the `Retriever` class from `src.rag` with **hybrid scoring enabled**.

In [5]:
# Create Retriever with hybrid scoring
retriever = Retriever(
    knowledge_base=knowledge_base,
    top_k=5,
    similarity_threshold=0.3,   # Lower threshold to see more results
    topic_boost=0.15,           # Boost per matching topic
    use_hybrid_scoring=True,    # Enable topic boosting
)

print("✅ Retriever initialized with HYBRID SCORING enabled!")
print(f"   Topic boost per match: {retriever.topic_boost}")
print(f"   Similarity threshold: {retriever.similarity_threshold}")

2026-03-26 01:03:36.683 | INFO     | src.rag.retriever:__init__:170 - Initialized Retriever with top_k=5, threshold=0.3, topic_boost=0.15, hybrid_scoring=True


✅ Retriever initialized with HYBRID SCORING enabled!
   Topic boost per match: 0.15
   Similarity threshold: 0.3


## 5. Test Retrieval with Sample Prompts

Using the `Retriever.retrieve_with_metadata()` method which applies hybrid scoring automatically.

In [6]:
# Define test prompts
test_prompts = [
    "Create a Bell state circuit with two qubits",
    "Implement Grover's search algorithm",
    "Build a QAOA circuit for MaxCut optimization",
    "Quantum phase estimation example",
    "How to implement quantum teleportation",
    "bell_state entanglement",  # Topic-focused query
    "qpe phase_estimation",     # Topic-focused query
]

print(f"Testing with {len(test_prompts)} prompts...")
print("📌 Using HYBRID SCORING from src.rag.Retriever!\n")

Testing with 7 prompts...
📌 Using HYBRID SCORING from src.rag.Retriever!



In [7]:
# Run tests for each prompt
for i, prompt in enumerate(test_prompts, 1):
    print("=" * 80)
    print(f"Test {i}: \"{prompt}\"")
    print("=" * 80)
    
    # Show keywords extracted
    keywords = extract_query_keywords(prompt)
    print(f"Query Keywords: {keywords}")
    
    # Use Retriever from src.rag
    results = retriever.retrieve_with_metadata(prompt, top_k=5)
    
    if results:
        print(f"\nTop 5 Retrieved Entries (Hybrid Scoring):")
        for rank, res in enumerate(results, 1):
            entry = res.get('entry', {})
            entry_id = entry.get('id', res.get('id', 'N/A'))
            hybrid_score = res.get('score', 0)
            semantic_score = res.get('semantic_score', hybrid_score)
            topic_boost = res.get('topic_boost', 0)
            difficulty = entry.get('difficulty', 'N/A')
            topics = entry.get('topics', [])
            task_preview = entry.get('task', entry.get('description', 'N/A'))[:60]
            
            boost_str = f" +{topic_boost:.2f}" if topic_boost > 0 else ""
            print(f"\n  #{rank} [Score: {hybrid_score:.4f} = {semantic_score:.4f}{boost_str}]")
            print(f"      ID: {entry_id}")
            print(f"      Difficulty: {difficulty}")
            print(f"      Topics: {', '.join(topics[:5]) if topics else 'N/A'}")
            print(f"      Task: {task_preview}...")
    else:
        print("  No results found.")
    
    print()

2026-03-26 01:03:36.702 | DEBUG    | src.rag.embeddings:encode:227 - Generating embeddings for 1 texts (provider=aws)


Test 1: "Create a Bell state circuit with two qubits"
Query Keywords: {'circuit', 'qubits', 'create', 'two', 'bell', 'state', 'with'}


2026-03-26 01:03:37.139 | DEBUG    | src.rag.retriever:retrieve:277 - Retrieved 5 results for query: Create a Bell state circuit with two qubits... (hybrid=True)
2026-03-26 01:03:37.141 | DEBUG    | src.rag.embeddings:encode:227 - Generating embeddings for 1 texts (provider=aws)



Top 5 Retrieved Entries (Hybrid Scoring):

  #1 [Score: 1.2112 = 0.7612 +0.45]
      ID: val_bell_state_00_11
      Difficulty: beginner
      Topics: bell_state, entanglement, two_qubit, validation
      Task: Creates the Bell state |Φ+⟩ = (|00⟩ + |11⟩)/√2. After measur...

  #2 [Score: 1.2077 = 0.7577 +0.45]
      ID: val_bell_state_01_10
      Difficulty: beginner
      Topics: bell_state, entanglement, two_qubit, validation
      Task: Creates the Bell state |Ψ+⟩ = (|01⟩ + |10⟩)/√2. After measur...

  #3 [Score: 1.0705 = 0.7705 +0.30]
      ID: design_bell_state
      Difficulty: beginner
      Topics: bell_state, entanglement, hadamard, cnot
      Task: Design a Cirq circuit that prepares the Bell state (|00> + |...

  #4 [Score: 1.0193 = 0.7193 +0.30]
      ID: design_bell_state_v3
      Difficulty: beginner
      Topics: bell_state, entanglement
      Task: Design a Bell-state circuit that uses an explicit Moment str...

  #5 [Score: 1.0175 = 0.7175 +0.30]
      ID: design_bell

2026-03-26 01:03:37.570 | DEBUG    | src.rag.retriever:retrieve:277 - Retrieved 5 results for query: Implement Grover's search algorithm... (hybrid=True)
2026-03-26 01:03:37.571 | DEBUG    | src.rag.embeddings:encode:227 - Generating embeddings for 1 texts (provider=aws)


...


Top 5 Retrieved Entries (Hybrid Scoring):

  #1 [Score: 0.8259 = 0.6759 +0.15]
      ID: val_teleportation
      Difficulty: advanced
      Topics: teleportation, entanglement, three_qubit, validation
      Task: Teleports state from qubit 0 to qubit 2 using entangled pair...

  #2 [Score: 0.8092 = 0.6592 +0.15]
      ID: designer_teleportation_circuit
      Difficulty: intermediate
      Topics: teleportation, entanglement, bell_measurement
      Task: Implement the Quantum Teleportation circuit to transfer a st...

  #3 [Score: 0.7848 = 0.6348 +0.15]
      ID: designer_teleportation_simulation
      Difficulty: intermediate
      Topics: teleportation, simulation, bloch_sphere
      Task: Simulate the teleportation circuit and verify the state tran...

  #4 [Score: 0.7824 = 0.6324 +0.15]
      ID: design_quantum_walk_1d_coin
      Difficulty: intermediate
      Topics: quantum_walk, coin, superposition
      Task: Implement a single step of a 1D quantum walk on a line using...

  #5

2026-03-26 01:03:39.220 | DEBUG    | src.rag.retriever:retrieve:277 - Retrieved 5 results for query: bell_state entanglement... (hybrid=True)
2026-03-26 01:03:39.221 | DEBUG    | src.rag.embeddings:encode:227 - Generating embeddings for 1 texts (provider=aws)



Top 5 Retrieved Entries (Hybrid Scoring):

  #1 [Score: 0.9466 = 0.6466 +0.30]
      ID: val_bell_state_00_11
      Difficulty: beginner
      Topics: bell_state, entanglement, two_qubit, validation
      Task: Creates the Bell state |Φ+⟩ = (|00⟩ + |11⟩)/√2. After measur...

  #2 [Score: 0.9453 = 0.6453 +0.30]
      ID: val_bell_state_01_10
      Difficulty: beginner
      Topics: bell_state, entanglement, two_qubit, validation
      Task: Creates the Bell state |Ψ+⟩ = (|01⟩ + |10⟩)/√2. After measur...

  #3 [Score: 0.9378 = 0.6378 +0.30]
      ID: design_bell_state
      Difficulty: beginner
      Topics: bell_state, entanglement, hadamard, cnot
      Task: Design a Cirq circuit that prepares the Bell state (|00> + |...

  #4 [Score: 0.9264 = 0.6264 +0.30]
      ID: design_bell_state_v3
      Difficulty: beginner
      Topics: bell_state, entanglement
      Task: Design a Bell-state circuit that uses an explicit Moment str...

  #5 [Score: 0.9084 = 0.6084 +0.30]
      ID: design_bell

2026-03-26 01:03:39.629 | DEBUG    | src.rag.retriever:retrieve:277 - Retrieved 5 results for query: qpe phase_estimation... (hybrid=True)



Top 5 Retrieved Entries (Hybrid Scoring):

  #1 [Score: 0.9900 = 0.6900 +0.30]
      ID: design_qpe_general
      Difficulty: advanced
      Topics: qpe, phase_estimation, general
      Task: Create a general Quantum Phase Estimation function in Cirq t...

  #2 [Score: 0.9867 = 0.6867 +0.30]
      ID: design_qpe_rz
      Difficulty: advanced
      Topics: qpe, phase_estimation, rz_gate
      Task: Estimate the phase of an Rz(theta) gate where theta=pi/3 usi...

  #3 [Score: 0.9833 = 0.6833 +0.30]
      ID: design_qpe_t_gate
      Difficulty: advanced
      Topics: qpe, phase_estimation, t_gate
      Task: Estimate the phase of a T gate (phase = 1/8) using 2 countin...

  #4 [Score: 0.9737 = 0.6737 +0.30]
      ID: designer_quantum_phase_estimation_t_gate
      Difficulty: advanced
      Topics: qpe, phase_estimation, t_gate
      Task: Estimate the phase of a T gate (phase pi/4) using 3 precisio...

  #5 [Score: 0.8147 = 0.6647 +0.15]
      ID: designer_shor_order_finding_circuit
    

## 6. Save Vector Store

In [8]:
# Save the index
knowledge_base.save_index(VECTOR_INDEX_PATH)

print(f"✅ Vector store saved to: {VECTOR_INDEX_PATH}")
print(f"   Total entries indexed: {knowledge_base.vector_store.size()}")

2026-03-26 01:03:39.640 | INFO     | src.rag.vector_store:save:435 - Saved FAISS index to D:\University\Uni\Huawei ICT Compeition\Cirq-RAG-Code-Assistant\data\models\vector_index
2026-03-26 01:03:39.641 | INFO     | src.rag.knowledge_base:save_index:375 - Saved knowledge base index to D:\University\Uni\Huawei ICT Compeition\Cirq-RAG-Code-Assistant\data\models\vector_index


✅ Vector store saved to: D:\University\Uni\Huawei ICT Compeition\Cirq-RAG-Code-Assistant\data\models\vector_index
   Total entries indexed: 140


## 7. Verify Load

In [9]:
# Create new KnowledgeBase and load from disk
loaded_kb = KnowledgeBase(knowledge_base_path=KNOWLEDGE_BASE_DIR)
loaded_kb.load_from_directory()  # Load entries first
loaded_kb.load_index(VECTOR_INDEX_PATH)  # Then load vector index

# Create Retriever
loaded_retriever = Retriever(
    knowledge_base=loaded_kb,
    use_hybrid_scoring=True,
    similarity_threshold=0.3,
)

print(f"✅ Loaded from disk.")
print(f"   Entries: {len(loaded_kb.entries)}")
print(f"   Vector store size: {loaded_kb.vector_store.size()}")

# Test
test_query = "Create a bell state circuit"
print(f"\nVerification Query: \"{test_query}\"")
print(f"Query Keywords: {extract_query_keywords(test_query)}")
print(f"\nTop 5 Results (Hybrid Scoring):")

verify_results = loaded_retriever.retrieve(test_query, top_k=5)
for res in verify_results:
    entry = res.get('entry', {})
    topics = entry.get('topics', [])
    boost = res.get('topic_boost', 0)
    boost_str = f" [+{boost:.2f} boost]" if boost > 0 else ""
    print(f"  - {entry.get('id', 'N/A')} (Score: {res['score']:.4f}){boost_str} | Topics: {', '.join(topics[:4]) if topics else 'N/A'}")

2026-03-26 01:03:39.654 | INFO     | src.rag.embeddings:_init_aws:124 - Using AWS Bedrock Embeddings: amazon.nova-2-multimodal-embeddings-v1:0, dim=1024
2026-03-26 01:03:39.655 | INFO     | src.rag.vector_store:_init_faiss:139 - Initialized FAISS index
2026-03-26 01:03:39.656 | INFO     | src.rag.vector_store:__init__:120 - Initialized VectorStore with faiss backend
2026-03-26 01:03:39.656 | INFO     | src.rag.vector_store:__init__:121 - Embedding dimension: 1024
2026-03-26 01:03:39.657 | INFO     | src.rag.knowledge_base:__init__:100 - Initialized KnowledgeBase with 0 entries
2026-03-26 01:03:39.657 | INFO     | src.rag.knowledge_base:load_from_jsonl:116 - Loading knowledge base from D:\University\Uni\Huawei ICT Compeition\Cirq-RAG-Code-Assistant\data\knowledge_base\curated_designer_examples.jsonl
2026-03-26 01:03:39.659 | INFO     | src.rag.knowledge_base:load_from_jsonl:129 - Loaded 107 entries from D:\University\Uni\Huawei ICT Compeition\Cirq-RAG-Code-Assistant\data\knowledge_base\

✅ Loaded from disk.
   Entries: 140
   Vector store size: 140

Verification Query: "Create a bell state circuit"
Query Keywords: {'circuit', 'bell', 'state', 'create'}

Top 5 Results (Hybrid Scoring):


2026-03-26 01:03:41.081 | DEBUG    | src.rag.retriever:retrieve:277 - Retrieved 5 results for query: Create a bell state circuit... (hybrid=True)


  - design_bell_state (Score: 0.9363) [+0.30 boost] | Topics: bell_state, entanglement, hadamard, cnot
  - design_bell_state_v3 (Score: 0.9314) [+0.30 boost] | Topics: bell_state, entanglement
  - val_bell_state_00_11 (Score: 0.9261) [+0.30 boost] | Topics: bell_state, entanglement, two_qubit, validation
  - val_bell_state_01_10 (Score: 0.9234) [+0.30 boost] | Topics: bell_state, entanglement, two_qubit, validation
  - design_bell_state_v2 (Score: 0.9130) [+0.30 boost] | Topics: bell_state, entanglement, hadamard, cnot


In [10]:
# =============================================================================
# Test Retrieval for VALIDATOR and OPTIMIZER Agents
# =============================================================================
print("=" * 80)
print("VALIDATOR & OPTIMIZER AGENT RETRIEVAL TESTS")
print("=" * 80)
print()

# Test prompts for Validator (looking for validation examples with expected outputs)
validator_prompts = [
    "validate bell state circuit expected output",
    "validate grover algorithm correctness",
    "check quantum teleportation measurement results",
]

# Test prompts for Optimizer (looking for optimization patterns)
optimizer_prompts = [
    "optimize circuit reduce depth gates",
    "cancel consecutive gates optimization",
    "merge single qubit gates PhasedXZ",
]

print("🔍 VALIDATOR AGENT RETRIEVAL TESTS")
print("-" * 50)
for i, prompt in enumerate(validator_prompts, 1):
    print(f"\n📌 Validator Query {i}: \"{prompt}\"")
    keywords = extract_query_keywords(prompt)
    print(f"   Keywords: {keywords}")
    
    results = retriever.retrieve_with_metadata(prompt, top_k=3)
    for rank, res in enumerate(results, 1):
        entry = res.get('entry', {})
        entry_id = entry.get('id', 'N/A')
        topics = entry.get('topics', [])
        score = res.get('score', 0)
        boost = res.get('topic_boost', 0)
        # Check if it's a validation example
        has_expected = "expected_output" in entry or "acceptable_ranges" in entry
        marker = "✅ VALIDATION" if has_expected else ""
        print(f"   #{rank} [{score:.4f}] {entry_id} | Topics: {', '.join(topics[:3])} {marker}")

print("\n")
print("⚡ OPTIMIZER AGENT RETRIEVAL TESTS")
print("-" * 50)
for i, prompt in enumerate(optimizer_prompts, 1):
    print(f"\n📌 Optimizer Query {i}: \"{prompt}\"")
    keywords = extract_query_keywords(prompt)
    print(f"   Keywords: {keywords}")
    
    results = retriever.retrieve_with_metadata(prompt, top_k=3)
    for rank, res in enumerate(results, 1):
        entry = res.get('entry', {})
        entry_id = entry.get('id', 'N/A')
        topics = entry.get('topics', [])
        score = res.get('score', 0)
        # Check if it's an optimization example
        has_optimization = "optimized_code" in entry or "optimization_type" in entry
        marker = "✅ OPTIMIZATION" if has_optimization else ""
        print(f"   #{rank} [{score:.4f}] {entry_id} | Topics: {', '.join(topics[:3])} {marker}")

print("\n" + "=" * 80)
print("✅ All agent retrieval tests completed!")
print("=" * 80)

2026-03-26 01:03:41.093 | DEBUG    | src.rag.embeddings:encode:227 - Generating embeddings for 1 texts (provider=aws)


VALIDATOR & OPTIMIZER AGENT RETRIEVAL TESTS

🔍 VALIDATOR AGENT RETRIEVAL TESTS
--------------------------------------------------

📌 Validator Query 1: "validate bell state circuit expected output"
   Keywords: {'circuit', 'expected', 'validate', 'output', 'bell', 'state'}


2026-03-26 01:03:41.541 | DEBUG    | src.rag.retriever:retrieve:277 - Retrieved 3 results for query: validate bell state circuit expected output... (hybrid=True)
2026-03-26 01:03:41.542 | DEBUG    | src.rag.embeddings:encode:227 - Generating embeddings for 1 texts (provider=aws)


   #1 [0.9198] val_bell_state_01_10 | Topics: bell_state, entanglement, two_qubit ✅ VALIDATION
   #2 [0.9171] val_bell_state_00_11 | Topics: bell_state, entanglement, two_qubit ✅ VALIDATION
   #3 [0.9133] design_bell_state_v2 | Topics: bell_state, entanglement, hadamard 

📌 Validator Query 2: "validate grover algorithm correctness"
   Keywords: {'algorithm', 'validate', 'correctness', 'grover'}


2026-03-26 01:03:41.952 | DEBUG    | src.rag.retriever:retrieve:277 - Retrieved 3 results for query: validate grover algorithm correctness... (hybrid=True)
2026-03-26 01:03:41.953 | DEBUG    | src.rag.embeddings:encode:227 - Generating embeddings for 1 texts (provider=aws)


...

   #1 [0.9341] opt_parallel_gates | Topics: optimization, parallelism, depth ✅ OPTIMIZATION
   #2 [0.8506] opt_grover_diffuser_simplify | Topics: optimization, grover, diffuser ✅ OPTIMIZATION
   #3 [0.7233] opt_circuit_stratified | Topics: optimization, stratify, layering ✅ OPTIMIZATION

📌 Optimizer Query 2: "cancel consecutive gates optimization"
   Keywords: {'consecutive', 'cancel', 'gates', 'optimization'}


2026-03-26 01:03:43.209 | DEBUG    | src.rag.retriever:retrieve:277 - Retrieved 3 results for query: cancel consecutive gates optimization... (hybrid=True)
2026-03-26 01:03:43.210 | DEBUG    | src.rag.embeddings:encode:227 - Generating embeddings for 1 texts (provider=aws)


   #1 [0.8468] opt_consecutive_x_cancel | Topics: optimization, gate_cancellation, x_gate ✅ OPTIMIZATION
   #2 [0.8260] opt_consecutive_h_cancel | Topics: optimization, gate_cancellation, hadamard ✅ OPTIMIZATION
   #3 [0.8199] opt_repeated_cz_cancel | Topics: optimization, cz, gate_cancellation ✅ OPTIMIZATION

📌 Optimizer Query 3: "merge single qubit gates PhasedXZ"
   Keywords: {'qubit', 'gates', 'merge', 'phasedxz', 'single'}


2026-03-26 01:03:43.641 | DEBUG    | src.rag.retriever:retrieve:277 - Retrieved 3 results for query: merge single qubit gates PhasedXZ... (hybrid=True)


   #1 [1.2859] opt_single_qubit_merge | Topics: optimization, gate_merge, single_qubit ✅ OPTIMIZATION
   #2 [0.8933] opt_rx_to_phxz | Topics: optimization, rotation, native_gates ✅ OPTIMIZATION
   #3 [0.8834] opt_toffoli_decompose | Topics: optimization, toffoli, decomposition ✅ OPTIMIZATION

✅ All agent retrieval tests completed!


## Summary

This notebook has:
1. ✅ Loaded entries using `KnowledgeBase.load_from_directory()`
2. ✅ Indexed entries using `KnowledgeBase.index_entries()`
3. ✅ Used `Retriever` from `src.rag` with **hybrid scoring**
4. ✅ Tested retrieval with sample prompts
5. ✅ Saved and loaded the index

### Hybrid Scoring (from src.rag.retriever)
```python
Hybrid Score = Semantic Score + (TOPIC_BOOST × Matching Topic Count)

# Default: TOPIC_BOOST = 0.15
```

Entries with matching topic keywords get a significant boost in ranking!